In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sympy.physics.wigner import wigner_3j, wigner_6j
from scipy import linalg
from sympy import Matrix
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment

In [2]:
#some constants
hbar = 6.62607015*10**-34 / (2*np.pi) # Joules seconds
h = hbar*2*np.pi # J seconds
eps0 = 8.8541878188e-12 
m_e = 9.1093837e-31 #kg
q_e = 1.60217663e-19 #C
a0 = 4*np.pi*eps0*hbar**2/(m_e*q_e**2)
u_B = 9.274010e-24 # J/T

In [3]:
def get_L_term(L, F, F_p, M, M_p, J, I, S):
    power_term = (-1)**(J+L+S+1+F_p+J+I+1+F-M)
    sqrt_terms = np.sqrt((2*F+1)*(2*F_p+1))*np.sqrt(L*(L+1)*(2*L+1))
    return power_term*(2*J+1)*sqrt_terms*hbar*wigner_3j(F,1,F_p,-M,0,M_p)*wigner_6j(J,F_p,I,F,J,1)*wigner_6j(L,J,S,J,L,1)

def get_S_term(S, F, F_p, M, M_p, J, I, L):
    power_term = (-1)**(F-M+F_p+J+I+J+S+L+1+1)
    sqrt_terms = np.sqrt((2*F+1)*(2*F_p+1))*np.sqrt((2*J+1)*(2*J+1))*np.sqrt(S*(S+1)*(2*S+1))
    return power_term*sqrt_terms*hbar*wigner_3j(F,1,F_p,-M,0,M_p)*wigner_6j(J,F_p,I,F,J,1)*wigner_6j(S,J,L,J,S,1)

def get_I_term(I, F, F_p, M, M_p, J):
    power_term = (-1)**(F_p+I+J+F-M+1)
    sqrt_terms = np.sqrt((2*F+1)*(2*F_p+1))*np.sqrt(I*(I+1)*(2*I+1))
    return power_term*sqrt_terms*hbar*wigner_3j(F,1,F_p,-M,0,M_p)*wigner_6j(I,F_p,J,F,I,1)

In [4]:
# for a given energy (M, M') in the matrix
def get_matrix_element(M, M_p, F, F_p, I, J, L, S, B, A_hfs, B_hfs):
    hyperfine_element = 0
    K = F*(F+1)-I*(I+1)-J*(J+1)
    if (F == F_p) & (M==M_p):
        hyperfine_element += A_hfs/(hbar**2)*K*hbar**2/2 
        if B_hfs != 0:
            hyperfine_element += B_hfs/(hbar**2)*(3/2*K*(K+1)-2*I*(I+1)*J*(J+1))/(4*I*(2*I-1)*J*(2*J-1))*hbar**2
       
    # Values from Steck (unitless)
    g_L = 0.99999339
    g_S = 2.002319304
    g_I = -0.0009951414

    L_term = -u_B*B/hbar*g_L*get_L_term(L, F, F_p, M, M_p, J, I, S)
    S_term = -u_B*B/hbar*g_S*get_S_term(S, F, F_p, M, M_p, J, I, L)
    I_term = -u_B*B/hbar*g_I*get_I_term(I, F, F_p, M, M_p, J)

    spherical_elements = L_term + S_term + I_term

    return spherical_elements + hyperfine_element

In [5]:
def create_FM_arrays(I,J):
    F_values = np.arange(abs(I-J),I+J+1) # F = |I-J| to |I+J|
    F_array, M_array = [], []
    for F in F_values:
        F_array.extend([F] * int(2*F+1))
        M_array.extend(np.arange(-F, F+1))  # works for both integer and half-integer F
    return np.array(F_array), np.array(M_array)

In [6]:
def get_H(size, Fv, F_pv, Mv, M_pv, I, J, L, S, B_val, A_hfs, B_hfs):
    H = np.zeros((size, size))
    for i in range(0, size):
        for j in range(0, size):
            F, F_p = Fv[i,j], F_pv[i,j]
            M, M_p = Mv[i,j], M_pv[i,j]
            H[i][j] = get_matrix_element(M, M_p, F, F_p, I, J, L, S, B_val, A_hfs, B_hfs)
    return H

In [7]:
def get_Zeeman_shift(state_params):
    A_hfs = state_params['A_hfs']
    B_hfs = state_params['B_hfs']
    B = state_params['B']
    J = state_params['J']
    L = state_params['L']
    I = state_params['I']
    S = state_params['S']

    F, M = create_FM_arrays(I,J)
    size = len(F)
    M_pv, Mv = np.meshgrid(M, M)
    F_pv, Fv = np.meshgrid(F, F)
    #print(M_pv)
    #print(Mv)
    #print(F_pv)
    #print(Fv)

    eig_arr = []
    evec_arr = []
    evecs_old = None
    for i in tqdm(range(0, len(B))):
        B_val = B[i]
        H = get_H(size, Fv, F_pv, Mv, M_pv, I, J, L, S, B_val, A_hfs, B_hfs)
        eigenvalues, eigenvectors = linalg.eigh(H)

        if i == 0:
            eig_arr.append(eigenvalues)
            evec_arr.append(eigenvectors)
            evecs_old = np.copy(eigenvectors)
        else:
            overlap = np.abs(evecs_old.T @ eigenvectors)
            cost_matrix = 1 - overlap # 1-|<evecs_old|eigenvectors>|
            _, col_indices = linear_sum_assignment(cost_matrix)

            eig_arr.append(eigenvalues[col_indices])
            evec_arr.append(eigenvectors[col_indices])
            evecs_old = eigenvectors[:, col_indices]
    return np.array(eig_arr), np.array(evec_arr)

In [37]:
#From Steck Rb87 data
def transition_dipole_matrix_element(L,J,Lp,Jp):
    if L==0 and J==1/2 and Lp==1 and Jp== 3/2:
        return 4.227*q_e*a0 #table 7
    elif L==0 and J==1/2 and Lp==1 and Jp== 1/2:
        return 2.992*q_e*a0 #table 7
    else:
        raise ValueError("Don't know this matrix element")

#from eq 2.78 of Nick Hutzler's 137a notes
def no_B_matrix_element(L,J,F,M,q,Lp,Jp,Fp,Mp):
    #assumptions for Rb87
    I = 3/2
    Ip= I
    S = 1/2
    Sp = S

    parity = (-1)**(F + Fp + J + Jp + Ip + L + Sp - M)
    mere_integral = transition_dipole_matrix_element(L,J,Lp,Jp)
    wigner_eckart = wigner_3j(F,1,Fp,-M,q,Mp) #2.75
    first_spectator = ((2*F+1)*(2*Fp+1))**(1/2)*wigner_6j(Jp,Fp,I,F,J,1) #2.76
    second_spectator = ((2*J+1)*(2*Jp+1))**(1/2)*wigner_6j(Lp,Jp,S,J,L,1) #2.77

    return parity*mere_integral*wigner_eckart*first_spectator*second_spectator

In [9]:
#inputs: transition from an S state to a P state and a B field strength
# |L=0,J=1/2,F,M> -> |L=1,J,F,M> , B
# Calculate the eigenvectors under the given magnetic field
# The total transition strength is the double sum over all the transition strengths between the unperturbed eigenstates,
# weighted by the coefficients
# for now we are going to pass the B field strength as an index due to the way the big arrays are constructed

B = np.linspace(0, 10000/10000, 1000)

S_12 = {
    'J': 1/2,
    'L': 0,
    'A_hfs': h*3.417341e9, # J
    'B_hfs': 0, # J
    'g_J': 2.00233113,
    'g_I': -0.0009951414,
    'B': B, # T, for ground state 5S_1/2
    'I': 3/2,
    'S': 1/2,
}

# 5²P₁/₂ D1 excited state parameters
P_12 = {
    'J': 1/2,
    'L': 1,
    'A_hfs': h*407.25e6,  # J
    'B_hfs': 0, # J
    'g_J': 0.666,
    'g_I': -0.0009951414,
    'B': B, # T, for excited state 5P_1/2
    'I': 3/2,
    'S': 1/2,
}

# 5²P₃/₂ D2 excited state parameters
P_32 = {
    'J': 3/2,
    'L': 1,
    'A_hfs': h*84.7185e6, # J
    'g_J': 1.3362,
    'g_I': -0.0009951414,
    'B': B, # T, for excited state 5P_3/2
    'B_hfs': h*12.4965e6, # J, for 2P_3/2, include electric quadrupole term (no longer negligible)
    'I': 3/2,
    'S': 1/2,
}   

In [10]:
g_params = S_12
g_eig_arr, g_evec_arr = get_Zeeman_shift(g_params)
gF, gM = create_FM_arrays(g_params['I'],g_params['J'])

e_params = P_32
e_eig_arr, e_evec_arr = get_Zeeman_shift(e_params)
eF, eM = create_FM_arrays(e_params['I'],e_params['J'])

100%|██████████████████████████████████████████████████████████████████████████| 1000/1000 [07:12<00:00,  2.31it/s]


In [26]:
gnums = list(zip(gF,gM))
enums = list(zip(eF,eM))

In [31]:
np.size(gnums)

16

In [38]:
no_B_matrix_element_arr = np.zeros((np.size(gnums),np.size(enums)))
L = g_params['L']
J = g_params['J']
Lp = e_params['L']
Jp = e_params['J']
for ii,gnum in enumerate(gnums):
    for jj,enum in enumerate(enums):
        no_B_matrix_element_arr[ii,jj] = no_B_matrix_element(L,J,gnum[0],gnum[1],0,Lp,Jp,enum[0],enum[1])

In [39]:
no_B_matrix_element_arr

array([[ 0.00000000e+00, -1.33560344e-29,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00, -1.03455397e-29,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 1.19460003e-29,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        -1.19460003e-29,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00